# Volatility Regression 01 — full-feature, next-five-minute OLS

**Research question.** Can the full causal market/derivatives feature catalog predict **volatility in the next five-minute interval**, and which predictors can be removed without worsening future-sample forecasts?

- Target: next-bar five-minute **Parkinson volatility**, σ = log(high/low)/sqrt(4 log 2), derived from the NEXT perpetual trade OHLC bar. **Y is volatility, not returns or squared returns**. The regression uses log(σ + ε) for numerical conditioning; ε is derived from training data only.
- X: all eligible contemporaneous predictors in the existing `full_information.FEATURE_SPECS` catalog, plus the current 5m return, current bar range, clock terms and time to expiry. We explicitly inventory omitted, low-coverage and aliased features.
- Strict time alignment: X at boundary t predicts the five-minute bar ending t+5; no gap bridging, no session crossover; target bar must pass `perp_integrity_ok`.
- Chronological **development-only** train / selection / untouched-test splits. HSMM 06's prospective 30-session holdout remains untouched. Feature removal uses the selection split, never the test split.
- Evaluation: next-bar log-volatility RMSE/MAE and out-of-sample R². Baseline/full/reduced comparisons use exactly the same complete-case population.
- Caveats: trade-based Parkinson volatility is a noisy proxy affected by bid–ask bounce; current 5m feature frequency does **not** imply independent observations. Complete-case selection can become very sparse with option features; this notebook reports that instead of imputing.

Research-only notebook. Reusable logic belongs to `src/data_engineering/modeling/vol_ols_5m.py`. Do not interpret development results as confirmed trading alpha.

## 1. Environment / source data — DuckDB read only

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch JupyterLab from the repository root (or one of its subfolders).")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

from data_engineering.modeling import vol_ols_5m as mod
OUTPUT = ROOT / "notebooks" / "Volatility_Regression"
OUTPUT.mkdir(parents=True, exist_ok=True)

# Excludes post-2026-10-07 data reserved for HSMM 06 confirmation.
DEVELOPMENT_END = "2026-10-07"
COVERAGE_MIN = 0.50
MAX_DROPS = 15
MIN_GAIN = 0.0  # strictly positive selection-RMSE improvement required
ASSETS = ("BTC", "ETH", "SOL", "AVAX", "HYPE", "XRP")
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    df = con.execute("""
        SELECT *
        FROM time_series_feature_state
        WHERE TRY_CAST(expiry_date AS DATE) <= DATE '2026-10-07'
        ORDER BY underlying, state_time
    """).df()
print("Vol-OLS version:", mod.VOL_OLS_5M_VERSION)
print("Development rows:", len(df), "assets:", df["underlying"].nunique())
print("Data spans:", df.state_time.min(), "to", df.state_time.max())

In [ ]:
## Synthetic alignment and estimator tests — abort on failure.
import subprocess
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s",
     str(ROOT / "tests"), "-p", "test_vol_ols_5m.py", "-v"],
    cwd=ROOT, check=True
)

## 2. Fit broad OLS and perform backward elimination on an isolated selection split

In [ ]:
runs = []
stats = []
inventory_frames = []
coverage_frames = []
coefficient_frames = []
elimination_frames = []
errors = []
for asset in ASSETS:
    g = df[df["underlying"] == asset]
    if g.empty:
        errors.append({"underlying": asset, "reason": "No rows in development sample"})
        continue
    try:
        d = mod.prepare_asset(g, coverage=COVERAGE_MIN,
                              development_end=DEVELOPMENT_END)
        X, y = d["X"], d["y"]
        names = d["features"]
        full_common = np.isfinite(y) & np.isfinite(X[names].to_numpy(float)).all(axis=1)
        tr = d["train"] & full_common
        sel = d["selection"] & full_common
        tst = d["test"] & full_common
        if tr.sum() <= len(names) + 20 or sel.sum() < 20 or tst.sum() < 20:
            raise ValueError(f"Insufficient common rows: train={tr.sum()}, "
                             f"selection={sel.sum()}, test={tst.sum()}, p={len(names)}")
        full = mod.fit_ols(X, y, tr, names)
        reduced_names, hist = mod.backward_validation(
            X, y, tr, sel, names,
            max_drops=MAX_DROPS, min_improvement=MIN_GAIN
        )
        reduced = mod.fit_ols(X, y, tr, reduced_names)
        baseline_names = [n for n in (
            "log_rv_30m", "log_rv_1h", "log_rv_4h",
            "weekend", "clock_sin1", "clock_cos1",
            "clock_sin2", "clock_cos2", "log1p_tte_min"
        ) if n in names]
        if not baseline_names:
            raise ValueError("No viable HAR/clock baseline in the full feature set")
        baseline = mod.fit_ols(X, y, tr, baseline_names)
        for model_name, fit in (("baseline", baseline),
                                ("full_ols", full), ("reduced_ols", reduced)):
            for split, mask in (("train", tr), ("selection", sel), ("test", tst)):
                stats.append({"underlying": asset, "model": model_name,
                              "features": len(fit.names),
                              "condition_number": fit.condition_number,
                              "epsilon_train": d["epsilon"],
                              **mod.metrics(fit, X, y, mask, split)})
        for name, beta in zip(full.names, full.beta[1:]):
            coefficient_frames.append({"underlying": asset, "feature": name,
                                       "beta_standardized": float(beta)})
        inv = d["inventory"].copy()
        inv["underlying"] = asset
        inv["train_coverage"] = inv["name"].map(d["coverage"])
        inv["full_ols"] = inv["name"].isin(names)
        inv["low_coverage"] = inv["name"].isin(d["excluded_coverage"])
        inv["low_joint_coverage"] = inv["name"].isin(d["excluded_joint_coverage"])
        inv["aliased_or_constant"] = inv["name"].isin(d["excluded_aliases"])
        inventory_frames.append(inv)
        coverage_frames.append({"underlying": asset,
                                "eligible_target_train": int((d["train"] & np.isfinite(y)).sum()),
                                "complete_train": int(tr.sum()),
                                "complete_selection": int(sel.sum()),
                                "complete_test": int(tst.sum()),
                                "n_sessions": len(d["sessions"]),
                                "catalog_available": d["original_feature_count"],
                                "full_p": len(names), "reduced_p": len(reduced_names),
                                "max_drops_reached": len(hist) >= MAX_DROPS,
                                "removed_for_coverage": len(d["excluded_coverage"]),
                                "removed_for_joint_coverage": len(d["excluded_joint_coverage"]),
                                "removed_for_alias": len(d["excluded_aliases"])})
        if len(hist):
            hist["underlying"] = asset
            elimination_frames.append(hist)
        runs.append((asset, d, baseline, full, reduced))
        print(asset, "n:", int(tr.sum()), int(sel.sum()), int(tst.sum()),
              "features:", len(names), "->", len(reduced_names))
    except Exception as exc:
        errors.append({"underlying": asset, "reason": f"{type(exc).__name__}: {exc}"})
        print(asset, "NOT FIT:", errors[-1]["reason"])
metrics_df = pd.DataFrame(stats)
coverage_df = pd.DataFrame(coverage_frames)
inventory_df = pd.concat(inventory_frames, ignore_index=True) if inventory_frames else pd.DataFrame()
coefficients_df = pd.DataFrame(coefficient_frames)
elimination_df = pd.concat(elimination_frames, ignore_index=True) if elimination_frames else pd.DataFrame()
errors_df = pd.DataFrame(errors)
display(coverage_df)
display(metrics_df[metrics_df["split"] == "test"])
display(errors_df)

## 3. Coefficient stability, feature support and chronological scoring

In [ ]:
if not coefficients_df.empty:
    print("Largest absolute full-model standardized coefficients (descriptive only):")
    for asset, rows in coefficients_df.groupby("underlying"):
        print("\n", asset)
        display(rows.assign(abs_beta=rows.beta_standardized.abs())
                .nlargest(12, "abs_beta")[["feature", "beta_standardized"]])
if not elimination_df.empty:
    display(elimination_df)
if not inventory_df.empty:
    print("Complete predictor inventory, including unavailable columns and coverage exclusions:")
    display(inventory_df.groupby(["underlying", "full_ols", "low_coverage",
                                  "low_joint_coverage", "aliased_or_constant"], dropna=False).size())

## 4. Save reviewable outputs (no database writes, no frozen-spec changes)

In [ ]:
outputs = {
    "01_scores.csv": metrics_df,
    "01_coverage.csv": coverage_df,
    "01_feature_inventory.csv": inventory_df,
    "01_full_ols_coefficients.csv": coefficients_df,
    "01_elimination_history.csv": elimination_df,
    "01_fit_errors.csv": errors_df,
}
for name, table in outputs.items():
    path = OUTPUT / name
    table.to_csv(path, index=False)
    print(path.name, "rows:", len(table))
summary = {
    "version": mod.VOL_OLS_5M_VERSION,
    "target": "next_5m_parkinson_trade_range_volatility",
    "transformed_target": "log(sigma_next_5m + train_only_epsilon)",
    "development_last_expiry": DEVELOPMENT_END,
    "candidate_catalog": "full_information.FEATURE_SPECS + contemporaneous extras",
    "coverage_threshold": COVERAGE_MIN,
    "max_backward_drops": MAX_DROPS,
    "assets_fitted": [x[0] for x in runs],
    "assets_failed": errors_df.to_dict("records"),
    "claim": "development experiment only; no held-out confirmation",
}
import json
(OUTPUT / "01_run_metadata.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print("Output complete. Commit CSV/JSON files after a local run for Claude to review.")

## Interpretation contract

1. A feature dropped for **coverage** or **collinearity** is **not** evidence of zero predictive content. Keep its status separate from backward deletions.
2. Backward elimination uses selection data only, with a **fixed complete-case panel**. Untouched test scores are a one-time diagnostic, **not** material for additional tuning.
3. OLS coefficients can be unstable under multicollinearity; interpret them cautiously and review possible train-only HAC/session uncertainty in follow-up work.
4. The wide feature catalog may reduce complete-case coverage dramatically; failures are explicit. Future research can test nested feature families without imputation.
5. Next-bar Parkinson volatility is an **experimental trade-range proxy**, not the index-based volatility target of HSMM 06. Bid–ask bounce and OHLC integrity must be investigated before any economic interpretation.
6. Do not change HSMM 06's frozen specification, confirmation window or registered gates. This notebook intentionally scores development expiry sessions only.